# Здесб будем генерировать задачи используя новароченную модель ГПТ.

### Стоит ли использовать FAISS, добавить длругие источники и тд ? Хммм даже не знаю как это можно сделать, что бы ГПТ просто выучил немного матеши.

In [352]:
import json

from IPython.core.display import Latex
from fontTools.afmLib import readlines


In [353]:
tasks_path = "parsed_task_dir/algebra_6_parsed_task_from_md.jsonl"

In [354]:
tasks = []

with open(tasks_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        task = json.loads(line)
        task = json.loads(task) # превращаем строку в dict
        print(task)
        tasks.append(task)
        print(50*"+")

{'id': '1', 'original': 'Заменив звездочку соответствующей цифрой, заполните таблицу.\n\n|  | Числа, <br> делящиеся <br> на 2 | Числа, <br> делящиеся <br> на 5 | Числа, <br> делящиеся <br> на 10 | Числа, <br> делящиеся <br> на 3 | Числа, <br> делящиеся <br> на 9 |\n| :---: | :---: | :---: | :---: | :---: | :---: |\n| $49^{*}$ |  |  |  |  |  |\n| $83^{*}$ |  |  |  |  |  |', 'solution': None, 'answer': None, 'topic': 'Делимость натуральных чисел', 'difficulty': 'A', 'tags': ['делимость', 'натуральные числа', 'признаки делимости', 'таблица', 'заполнение', 'цифры', 'числа с пропусками'], 'type': 'Практическая', 'valid': True, 'remark': None}
++++++++++++++++++++++++++++++++++++++++++++++++++
{'id': 'II. Задача 2', 'original': 'Из чисел $1,2,6,7,13,15,20,41,49$ выпишите числа, имеющие:\n\n1) только один делитель;\n2) только два делителя;\n3) более двух делителей.', 'solution': None, 'answer': None, 'topic': 'Делимость натуральных чисел', 'difficulty': 'B', 'tags': ['делители', 'натуральные 

In [355]:
import openai
import os
from dotenv import load_dotenv

load_dotenv()                # подтянет OPENAI_API_KEY из .env
assert os.getenv("OPENAI_API_KEY"), "Не найден OPENAI_API_KEY"

client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])



In [356]:
SYSTEM_PROMPT = ("""
    Ты генератор учебных заданий. Получишь один пример задачи и создашь новую задачу
    того же предмета и уровня, но с другими числами и формулировкой (без копирования).
    Включи пошаговое короткое решение и конечный ответ.

    ВСЕ ФОРМУЛЫ В LaTeX ОБЯЗАТЕЛЬНО ДОЛЖНЫ БЫТЬ:
       - обёрнуты в `$...$` (inline)
       - `\frac`, `\cdot`, `\text` и прочее должны быть синтаксически корректны

    Верни только JSON-объект с полями:
    'question', 'solution', 'topic', 'answer', 'type', 'tags', 'difficulty'. Никаких комментариев вне JSON.
    """
)

In [357]:
import re


def clean_gpt_output(raw: str) -> str:
    # убираем обёртку ```json ... ```
    if raw.lstrip().startswith("```"):
        raw = re.sub(r"```json\s*", "", raw, flags=re.I).strip()
        raw = re.sub(r"```$", "", raw).strip()
    return raw

In [358]:
from typing import Any, Dict, Sequence
from collections.abc import Mapping


def _format_reference(ref: Mapping[str, str]) -> str:
    parts = [f"{k}: {ref[k]}" for k in ("question", "topic", "type", "solution", "answer", "tags", "difficulty") if k in ref]
    return "\n".join(parts)

def format_task_for_json(task: Mapping[str, str]) -> Any:
    task["original"] = str(task["original"]).replace("\\\\", "\\")

    if isinstance(task["solution"], str):
      task["solution"] = str(task["solution"]).replace("\\\\", "\\")

    return task

def generate_task(
    reference_task: Mapping[str, str] | str,
    *,
    subject: str | None = None,
    model: str = "gpt-4o",
    temperature: float = .9,
    seed: int | None = None,
) -> Dict[str, Any]:


    reference_task = format_task_for_json(reference_task)

    reference_block = (
        reference_task if isinstance(reference_task, str) else _format_reference(reference_task)
    )

    user_prompt = (
        (f"Предмет: {subject}. " if subject else "")
        + "Вот пример задачи. Создай новую задачу с решением и ответом:\n\n"
        + reference_block
    )
    messages: Sequence[Dict[str, str]] = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_prompt},
    ]
    resp = client.chat.completions.create(
        model=model,
        temperature=temperature,
        seed=seed,
        messages=messages,
    )
    raw = resp.choices[0].message.content.strip()
    try:
        raw = raw.replace("\\", "\\\\")
        raw = clean_gpt_output(raw)
        data = json.loads(raw)
        assert isinstance(data, dict)
    except Exception as exc:
        raise ValueError(f"Модель вернула невалидный JSON:\n{raw}") from exc
    return data

In [390]:
tasks

[{'id': '1',
  'original': 'Заменив звездочку соответствующей цифрой, заполните таблицу.\n\n|  | Числа, <br> делящиеся <br> на 2 | Числа, <br> делящиеся <br> на 5 | Числа, <br> делящиеся <br> на 10 | Числа, <br> делящиеся <br> на 3 | Числа, <br> делящиеся <br> на 9 |\n| :---: | :---: | :---: | :---: | :---: | :---: |\n| $49^{*}$ |  |  |  |  |  |\n| $83^{*}$ |  |  |  |  |  |',
  'solution': None,
  'answer': None,
  'topic': 'Делимость натуральных чисел',
  'difficulty': 'A',
  'tags': ['делимость',
   'натуральные числа',
   'признаки делимости',
   'таблица',
   'заполнение',
   'цифры',
   'числа с пропусками'],
  'type': 'Практическая',
  'valid': True,
  'remark': None},
 {'id': 'II. Задача 2',
  'original': 'Из чисел $1,2,6,7,13,15,20,41,49$ выпишите числа, имеющие:\n\n1) только один делитель;\n2) только два делителя;\n3) более двух делителей.',
  'solution': None,
  'answer': None,
  'topic': 'Делимость натуральных чисел',
  'difficulty': 'B',
  'tags': ['делители',
   'натуральн

In [360]:
type(tasks[0])

dict

In [361]:
new_task = generate_task(tasks[0], subject="Алгебра", model="gpt-4o-mini", seed=42)

In [362]:
new_task

{'question': 'Заполните таблицу, указав, делится ли число 84 на 7 и 12. Укажите, какое максимальное целое число, на которое делится 84, используя только цифры 0, 1, 2, ..., 9.',
 'solution': '1. Проверим делимость числа 84 на 7:\\\\n$84 \\\\div 7 = 12$, остаток 0. Значит, 84 делится на 7.\\\\n2. Проверим делимость числа 84 на 12:\\\\n$84 \\\\div 12 = 7$, остаток 0. Значит, 84 делится на 12.\\\\n3. Максимальное целое число, на которое делится 84, рассматривая доступные цифры, это 84, так как все цифры в числе 84 являются допустимыми.',
 'topic': 'Делимость натуральных чисел',
 'answer': '84 делится на 7 и 12; максимальное целое число - 84.',
 'type': 'Практическая',
 'tags': ['делимость',
  'натуральные числа',
  'признаки делимости',
  'таблица',
  'заполнение',
  'цифры',
  'числа с пропусками'],
 'difficulty': 'A'}

In [363]:
print(new_task["question"] + "\n" + new_task["solution"])

Заполните таблицу, указав, делится ли число 84 на 7 и 12. Укажите, какое максимальное целое число, на которое делится 84, используя только цифры 0, 1, 2, ..., 9.
1. Проверим делимость числа 84 на 7:\\n$84 \\div 7 = 12$, остаток 0. Значит, 84 делится на 7.\\n2. Проверим делимость числа 84 на 12:\\n$84 \\div 12 = 7$, остаток 0. Значит, 84 делится на 12.\\n3. Максимальное целое число, на которое делится 84, рассматривая доступные цифры, это 84, так как все цифры в числе 84 являются допустимыми.


Найдите все двузначные натуральные числа, которые делятся на 4 и у которых сумма цифр равна 8.


Всего 33 задач доступно

In [364]:
tasks[25]

{'id': '28',
 'original': 'Турист плыл на теплоходе $2 \\frac{1}{3}$ ч по течению реки, а затем 1,5 ч - по озеру. Собственная скорость теплохода $32 \\mathrm{\\kappa m} / \\mathrm{ч}$. Скорость течения реки 2,2 км $/$ ч. Какое расстояние проплыл турист за указанное время?',
 'solution': None,
 'answer': None,
 'topic': 'Сложные задачи на движение по реке',
 'difficulty': 'C',
 'tags': ['задача на движение',
  'скорость',
  'время',
  'расстояние',
  'течение реки',
  'теплоход'],
 'type': 'Практическая',
 'valid': True,
 'remark': None}

In [365]:
print(tasks[25]["original"])

Турист плыл на теплоходе $2 \frac{1}{3}$ ч по течению реки, а затем 1,5 ч - по озеру. Собственная скорость теплохода $32 \mathrm{\kappa m} / \mathrm{ч}$. Скорость течения реки 2,2 км $/$ ч. Какое расстояние проплыл турист за указанное время?


Турист плыл на теплоходе $2 \frac{1}{3}$ ч по течению реки, а затем 1,5 ч - по озеру. Собственная скорость теплохода $32 \mathrm{\kappa m} / \mathrm{ч}$. Скорость течения реки 2,2 км $/$ ч. Какое расстояние проплыл турист за указанное время?


In [366]:
new_task = generate_task(tasks[25], subject="Алгебра", model="gpt-4o-mini", seed=42)

In [367]:
print(new_task["question"].replace("\\\\", "\\"))
print(new_task["solution"].replace("\\\\", "\\"))

Два теплохода одновременно стартуют из одного и того же пункта и движутся по реке. Один теплоход движется со скоростью 18 км/ч, а другой - со скоростью 12 км/ч по течению. Расстояние между ними через 2 часа составит сколько километров?
1. Находим скорость относительного движения двух теплоходов: $18 + 12 = 30$ км/ч.\n2. Вычисляем расстояние, которое они прошли за 2 часа: $30 \cdot 2 = 60$ км.\nТак что расстояние между ними через 2 часа составит 60 км.


Сначала определим скорость теплохода против течения: \\(v_{против} = v_{теплохода} - v_{течения} = 20 - 5 = 15 \\text{ км/ч}\\).\\\nТеперь мы можем найти расстояние, пройденное теплоходом за 2 часа: \\(S = v_{против} \\cdot t = 15 \\cdot 2 = 30 \\text{ км}\\).

In [368]:
tasks[32]

{'id': '35',
 'original': 'Решите уравнение:\n1) $\\frac{1}{x}+\\frac{1}{1 \\frac{2}{7} x}=\\frac{4}{9}$;\n2) $\\frac{2}{1 \\frac{3}{5} x}-\\frac{1}{x}=\\frac{1}{12}$;\n3) $\\frac{2}{1 \\frac{1}{3} x}-\\frac{1}{x}=\\frac{1}{4}$.',
 'solution': None,
 'answer': None,
 'topic': 'Алгебраические выражения. Переменные',
 'difficulty': 'C',
 'tags': ['уравнение',
  'рациональные выражения',
  'переменная',
  'дроби',
  'решение уравнений'],
 'type': 'Практическая',
 'valid': True,
 'remark': None}

In [369]:
print(tasks[32]["original"])

Решите уравнение:
1) $\frac{1}{x}+\frac{1}{1 \frac{2}{7} x}=\frac{4}{9}$;
2) $\frac{2}{1 \frac{3}{5} x}-\frac{1}{x}=\frac{1}{12}$;
3) $\frac{2}{1 \frac{1}{3} x}-\frac{1}{x}=\frac{1}{4}$.


Решите уравнение:
1) $\frac{1}{x}+\frac{1}{1 \frac{2}{7} x}=\frac{4}{9}$;
2) $\frac{2}{1 \frac{3}{5} x}-\frac{1}{x}=\frac{1}{12}$;
3) $\frac{2}{1 \frac{1}{3} x}-\frac{1}{x}=\frac{1}{4}$.

In [370]:
new_task = generate_task(tasks[32], subject="Алгебра", model="gpt-4o-mini", seed=42)

In [371]:
print(new_task["question"].replace("\\\\", "\\"))
print(new_task["solution"].replace("\\\\", "\\"))

Решите уравнение: $\frac{2x - 3}{5} + 1 = \frac{x + 7}{3}$
1. Умножим обе стороны уравнения на 15 (наименьшее общее кратное знаменателей 5 и 3), чтобы избавиться от дробей:\n   $15 \cdot \left(\frac{2x - 3}{5} + 1\right) = 15 \cdot \frac{x + 7}{3}$\n   \n   2. После умножения получаем:\n   $3(2x - 3) + 15 = 5(x + 7)$\n   \n   3. Раскроем скобки:\n   $6x - 9 + 15 = 5x + 35$\n   \n   4. Упрощаем уравнение:\n   $6x + 6 = 5x + 35$\n   \n   5. Переносим $5x$ влево:\n   $6x - 5x = 35 - 6$\n   \n   6. Получаем:\n   $x = 29$


Решите уравнение: $\frac{2x + 3}{5} = \frac{x - 1}{2}$.

1. Умножим обе стороны уравнения на 10 для устранения дробей: \n   $10 \cdot \frac{2x + 3}{5} = 10 \cdot \frac{x - 1}{2}$ \n   $4x + 6 = 5(x - 1)$ \n\n2. Раскроем скобки: \n   $4x + 6 = 5x - 5$ \n\n3. Переносим все члены с переменной x в одну сторону, а свободные в другую: \n   $4x - 5x = -5 - 6$ \n   $-x = -11$ \n\n4. Умножаем обе стороны на -1: \n   $x = 11$.

In [372]:
print(tasks[28]["original"])

Сравните дроби:
1) 0,25 и $\frac{3}{4}$;
2) 0,5 и $\frac{1}{4}$;
3) $\frac{5}{6}$ и 0,4 ;
4) 0,6 и $\frac{4}{5}$;
5) 0,7 и $\frac{73}{100}$;
6) 0,2 и $\frac{9}{50}$.


Сравните дроби:
1) 0,25 и $\frac{3}{4}$;
2) 0,5 и $\frac{1}{4}$;
3) $\frac{5}{6}$ и 0,4 ;
4) 0,6 и $\frac{4}{5}$;
5) 0,7 и $\frac{73}{100}$;
6) 0,2 и $\frac{9}{50}$.

In [373]:
new_task = generate_task(tasks[28], subject="Алгебра", model="gpt-4o-mini", seed=42)

In [374]:
print(new_task["question"].replace("\\\\", "\\"))
print(new_task["solution"].replace("\\\\", "\\"))

Сравните числа $\frac{3}{7}$ и $0.5$. Какое из них больше?
Сначала преобразуем $0.5$ в дробь: $0.5 = \frac{1}{2}$. Теперь сравним дроби $\frac{3}{7}$ и $\frac{1}{2}$. Для этого найдём общий знаменатель, который равен $14$. Преобразуем дроби: $\frac{3}{7} = \frac{3 \cdot 2}{7 \cdot 2} = \frac{6}{14}$ и $\frac{1}{2} = \frac{1 \cdot 7}{2 \cdot 7} = \frac{7}{14}$. Теперь сравниваем: $\frac{6}{14} < \frac{7}{14}$, значит $\frac{3}{7} < 0.5$.


Сравните числа $\frac{3}{7}$ и $0.5$. Какое из них больше?

Сначала преобразуем $0.5$ в дробь: $0.5 = \frac{1}{2}$. Теперь сравним дроби $\frac{3}{7}$ и $\frac{1}{2}$. Для этого найдём общий знаменатель, который равен $14$. Преобразуем дроби: $\frac{3}{7} = \frac{3 \cdot 2}{7 \cdot 2} = \frac{6}{14}$ и $\frac{1}{2} = \frac{1 \cdot 7}{2 \cdot 7} = \frac{7}{14}$. Теперь сравниваем: $\frac{6}{14} < \frac{7}{14}$, значит $\frac{3}{7} < 0.5$.

In [375]:
# Пример чтения .jsonl файла, генерации и записи в новый файл
def process_jsonl_file(input_path, output_path):
    with open(input_path, 'r', encoding='utf-8') as infile, \
         open(output_path, 'w', encoding='utf-8') as outfile:

        for i, line in enumerate(infile, 1):
            try:
                task = json.loads(line)
                task = json.loads(task)
                new_task = generate_task(task, subject="Алгебра", model="gpt-4o", seed=42)
                outfile.write(json.dumps(new_task, ensure_ascii=False) + '\n')
                print(f"[{i}] Успешно сгенерировано")
            except Exception as e:
                print(f"[{i}] Ошибка: {e}")
                print(f"Ошибка обработки строки: {e}")

In [377]:
process_jsonl_file("parsed_task_dir/algebra_6_parsed_task_from_md.jsonl", "generated_tasks_dir/algebra_6_1_gen_parsed_tasks.jsonl")


[1] Успешно сгенерировано
[2] Успешно сгенерировано
[3] Успешно сгенерировано
[4] Успешно сгенерировано
[5] Успешно сгенерировано
[6] Успешно сгенерировано
[7] Успешно сгенерировано
[8] Успешно сгенерировано
[9] Успешно сгенерировано
[10] Успешно сгенерировано
[11] Успешно сгенерировано
[12] Успешно сгенерировано
[13] Успешно сгенерировано
[14] Успешно сгенерировано
[15] Успешно сгенерировано
[16] Успешно сгенерировано
[17] Успешно сгенерировано
[18] Успешно сгенерировано
[19] Успешно сгенерировано
[20] Успешно сгенерировано
[21] Успешно сгенерировано
[22] Успешно сгенерировано
[23] Успешно сгенерировано
[24] Успешно сгенерировано
[25] Успешно сгенерировано
[26] Успешно сгенерировано
[27] Успешно сгенерировано
[28] Успешно сгенерировано
[29] Успешно сгенерировано
[30] Успешно сгенерировано
[31] Успешно сгенерировано
[32] Успешно сгенерировано
[33] Успешно сгенерировано


In [378]:
new_tasks_path = "generated_tasks_dir/algebra_6_1_gen_parsed_tasks.jsonl"

In [379]:
new_tasks = []

with open(new_tasks_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        task = json.loads(line)
        #task = json.loads(task) # превращаем строку в dict
        print(task)
        new_tasks.append(task)
        print(50*"+")

{'question': 'Найдите наименьшее натуральное число, которое делится на 3, 4 и 5 одновременно.', 'solution': 'Чтобы найти наименьшее натуральное число, которое делится на 3, 4 и 5 одновременно, нужно найти наименьшее общее кратное (НОК) этих чисел. Разложим числа на простые множители:\\n\\n- 3 = 3\\n- 4 = 2^2\\n- 5 = 5\\n\\nНОК будет содержать каждый простой множитель в максимальной степени:\\n\\n$\\\\text{НОК} = 2^2 \\\\cdot 3 \\\\cdot 5 = 4 \\\\cdot 3 \\\\cdot 5 = 60$.\\n\\nНаименьшее число, которое делится на 3, 4 и 5 одновременно, равно 60.', 'topic': 'Делимость натуральных чисел', 'answer': '60', 'type': 'Практическая', 'tags': ['делимость', 'натуральные числа', 'НОК', 'простые множители'], 'difficulty': 'A'}
++++++++++++++++++++++++++++++++++++++++++++++++++
{'question': 'Найдите наибольший общий делитель (НОД) чисел 84 и 126.', 'solution': 'Мы начнем с разложения чисел на простые множители. \\n Для 84: \\n $84 = 2^2 \\\\cdot 3 \\\\cdot 7$.\\n Для 126: \\n $126 = 2 \\\\cdot 3^2 \\

In [327]:
print(new_tasks[10]["question"])

На чертеже изображен квадрат со стороной $3 \, \text{см}$, который соответствует площади $12 \, \text{м}^2$ в реальности. Какова площадь изображенного квадрата на чертеже в квадратных сантиметрах, если площадь на чертеже уменьшена в 25 раз по сравнению с реальностью?


На чертеже изображен квадрат со стороной $3 \, \text{см}$, который соответствует площади $12 \, \text{м}^2$ в реальности. Какова площадь изображенного квадрата на чертеже в квадратных сантиметрах, если площадь на чертеже уменьшена в 25 раз по сравнению с реальностью?

In [331]:
print(new_tasks[29]["question"])
print(new_tasks[29]["solution"])#.replace("\\\\","\\"))

Найдите сумму рациональных чисел: $2.5 + \frac{7}{4} - 1.75$.
Первый шаг: Преобразуйте все числа в десятичные дроби. \\n \\(2.5\\) остается как есть, \\(\\frac{7}{4} = 1.75\\), а \\(1.75\\) остается как есть. \\n Второй шаг: Сложите числа: \\(2.5 + 1.75 = 4.25\\). \\n Третий шаг: Вычтите последнее число: \\(4.25 - 1.75 = 2.5\\).


Найдите сумму рациональных чисел: $2.5 + \frac{7}{4} - 1.75$.

Первый шаг: Преобразуйте все числа в десятичные дроби. \\n \\(2.5\\) остается как есть, \\(\\frac{7}{4} = 1.75\\), а \\(1.75\\) остается как есть. \\n Второй шаг: Сложите числа: \\(2.5 + 1.75 = 4.25\\). \\n Третий шаг: Вычтите последнее число: \\(4.25 - 1.75 = 2.5\\).

In [329]:
print(new_tasks[28]["question"].replace("\\\\","\\"))
print(new_tasks[28]["solution"].replace("\\\\","\\"))

Сравните числа $0.75$ и $\frac{3}{4}$. Какое из них больше?
Переведем десятичную дробь $0.75$ в обыкновенную дробь. $0.75 = \frac{75}{100} = \frac{3}{4}$. Следовательно, $0.75 = \frac{3}{4}$. Оба числа равны.


Сравните числа $\frac{7}{5}$ и $1.4$. Какое из них больше?

Переведем дробь $\frac{7}{5}$ в десятичную форму. Для этого разделим 7 на 5:\n\n\ $[7 \div 5 = 1.4 ] $\n\nТеперь оба числа, $\frac{7}{5}$ и 1.4, равны и мы можем заключить, что они равны.\n\nИтак, $\frac{7}{5} = 1.4$.

In [330]:
tasks[28]

{'id': '31',
 'original': 'Сравните дроби:\n1) 0,25 и $\\frac{3}{4}$;\n2) 0,5 и $\\frac{1}{4}$;\n3) $\\frac{5}{6}$ и 0,4 ;\n4) 0,6 и $\\frac{4}{5}$;\n5) 0,7 и $\\frac{73}{100}$;\n6) 0,2 и $\\frac{9}{50}$.',
 'solution': None,
 'answer': None,
 'topic': 'Сравнение рациональных чисел',
 'difficulty': 'A',
 'tags': ['дроби',
  'сравнение',
  'десятичные дроби',
  'обыкновенные дроби',
  'числовые множества'],
 'type': 'Практическая',
 'valid': True,
 'remark': None}

Пусть \( n = 5k + 2 \) и \( n = 7m + 1 \), где \( k \) и \( m \) — целые числа.\n\nПриравняем выражения для \( n \):\n\n\\[ 5k + 2 = 7m + 1 \\]\n\nУпрощая уравнение, получим:\n\n\\[ 5k - 7m = -1 \\]\n\nТеперь необходимо найти такие \( k \) и \( m \), которые удовлетворяют этому уравнению. Попробуем подобрать подходящие значения.\n\nДля \( k = 3 \):\n\n\\[ 5 \cdot 3 - 7m = -1 \\]\n\\[ 15 - 7m = -1 \\]\n\\[ 7m = 16 \\]\n\nЭто значение \( m \) не является целым числом. Попробуем другие значения для \( k \).\n\nДля \( k = 4 \):\n\n\\[ 5 \cdot 4 - 7m = -1 \\]\n\\[ 20 - 7m = -1 \\]\n\\[ 7m = 21 \\]\n\\[ m = 3 \\]\n\nТеперь обе \( k = 4 \) и \( m = 3 \) целые числа. Проверим их в любом выражении для \( n \):\n\n\\[ n = 5 \cdot 4 + 2 = 22 \\]\n\nПроверка:\n\n- \( 22 \div 5 \) даёт остаток 2, \n- $ 22 \div 7 $ даёт остаток 1.\n\nТаким образом, подходящее число \( n = 22 \).

In [ ]:
import markdown  # pip install markdown
from pathlib import Path
from rich.console import Console
from rich.markdown import Markdown

In [335]:
def json_task_to_md(task: dict) -> str:
    """
    Принимает словарь с полями:
      task["question"], task["solution"], task.get("tags"), task.get("topic")
    Возвращает строку Markdown.
    """
    tags      = task["tags"]
    topic  = task["topic"]
    question   = task["question"].strip()
    solution = task["solution"].strip()

    md = (
    f"### {tags} ({topic})\n\n"
    f"**Условие**\n\n{question}\n\n"
    "---\n\n"
    f"**Решение**\n\n{solution}\n"
        )
    return md

In [ ]:
def validate_md(md_text: str) -> list[str]:
    """
    Возвращает список найденных проблем (если пустой — всё ок).
    Проверяем самые частые ошибки: непарные ``` и заголовки без пробела.
    """
    problems = []

    # 1) Непарные тройные бэктики
    if md_text.count("```") % 2 != 0:
        problems.append("непарное число ``` (кодовый блок не закрыт)")

    # 2) Заголовки вида '##НеЛезет'
    bad_heads = re.findall(r"^(#{1,6})(\S)", md_text, flags=re.MULTILINE)
    if bad_heads:
        problems.append("нет пробела после # в заголовке")

    # 3) markdown → HTML (если вылетит исключение, поймаем снаружи)
    _ = markdown.markdown(md_text, extensions=["fenced_code","tables"])

    return problems

In [348]:
from IPython.display import Markdown, display

display(Markdown(new_tasks[12]["question"].replace("\\\\", "\\")))

Упростите выражение: $\frac{x^2 - 4}{x^2 - 5x + 6} \cdot \frac{x - 3}{x + 2}$

Упростите выражение: $\frac{x^2 - 4}{x^2 - 5x + 6} \cdot \frac{x - 3}{x + 2}$

In [349]:
display(Markdown(new_tasks[7]["question"]))

Лодка движется по реке. Скорость течения реки составляет $3$ км/ч. Лодка проплывает $20$ км по течению и $20$ км против течения за $7$ часов. Определите собственную скорость лодки в стоячей воде.

In [380]:
for task in new_tasks:
    question = task["question"].strip().replace("\\\\", "\\")
    solution = task["solution"].strip().replace("\\\\", "\\")

    res = question + "\n\n" + solution + "\n" + 100*"~"
    display(Markdown(res))

Найдите наименьшее натуральное число, которое делится на 3, 4 и 5 одновременно.

Чтобы найти наименьшее натуральное число, которое делится на 3, 4 и 5 одновременно, нужно найти наименьшее общее кратное (НОК) этих чисел. Разложим числа на простые множители:\n\n- 3 = 3\n- 4 = 2^2\n- 5 = 5\n\nНОК будет содержать каждый простой множитель в максимальной степени:\n\n$\text{НОК} = 2^2 \cdot 3 \cdot 5 = 4 \cdot 3 \cdot 5 = 60$.\n\nНаименьшее число, которое делится на 3, 4 и 5 одновременно, равно 60.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Найдите наибольший общий делитель (НОД) чисел 84 и 126.

Мы начнем с разложения чисел на простые множители. \n Для 84: \n $84 = 2^2 \cdot 3 \cdot 7$.\n Для 126: \n $126 = 2 \cdot 3^2 \cdot 7$. \n Затем мы находим общие множители и перемножаем их с наименьшими показателями: \n Общий множитель для $2$: $2^1$, \n Общий множитель для $3$: $3^1$, \n Общий множитель для $7$: $7^1$. \n \n Перемножим общие множители: \n $2^1 \cdot 3^1 \cdot 7^1 = 42$. \n Таким образом, наибольший общий делитель (НОД) чисел 84 и 126 равен 42.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Определите, делится ли число 47325 на 3 и 9 без остатка.

Для проверки делимости числа на 3 необходимо найти сумму его цифр и проверить, делится ли она на 3. \[ 4 + 7 + 3 + 2 + 5 = 21 \] Поскольку 21 делится на 3, то число 47325 делится на 3.\n\nДля проверки делимости числа на 9 необходимо также найти сумму его цифр и проверить, делится ли она на 9. \[ 21 \] Поскольку 21 не делится на 9, число 47325 не делится на 9.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Докажите, что если натуральное число $a$ делится на 3, то квадрат этого числа $a^2$ также делится на 3.

Пусть $a$ - натуральное число, такое что $a \equiv 0 \pmod{3}$. Это означает, что существует некоторое целое число $k$, для которого $a = 3k$. Тогда $a^2 = (3k)^2 = 9k^2 = 3(3k^2)$. Поскольку $3k^2$ является целым числом, можно сделать вывод, что $a^2$ делится на 3.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Докажите, что для любого простого числа $p > 2$, число $p^2 - 1$ делится на 24.

Рассмотрим простое число $p$ больше 2. Тогда оно должно быть нечётным. Следовательно, $p$ можно представить в виде $p = 2k + 1$, где $k$ — целое число.\n\nТеперь найдём $p^2 - 1$:\n\n$$p^2 - 1 = (2k + 1)^2 - 1 = 4k^2 + 4k + 1 - 1 = 4k(k + 1).$$\n\nЗаметим, что произведение $k(k + 1)$ всегда чётно (одно из чисел обязательно чётное). Следовательно, $4k(k + 1)$ делится на 8.\n\nКроме того, среди трех последовательных чисел $p - 1$, $p$, $p + 1$ одно число делится на 3 (из-за делимости по модулю 3). Но числа $p - 1$ и $p + 1$ оба чётные и делятся на 2. Таким образом, хотя бы одно из произведений $(p - 1)/2$ и $(p + 1)/2$ делится на 3. Следовательно,  $p^2 - 1 = (p - 1)(p + 1)$ делится на 3.\n\nПоскольку $p^2 - 1$ делится на 8 и на 3, оно делится на их произведение, то есть на 24. \n\nТаким образом, доказано, что $p^2 - 1$ делится на 24.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

...

Какое из двух чисел больше: $\frac{7}{4}$ или $1.75$?

Для сравнения числа $\frac{7}{4}$ и $1.75$, преобразуем $\frac{7}{4}$ в десятичную дробь. Разделим 7 на 4, получим $7 \div 4 = 1.75$. Таким образом, $\frac{7}{4} = 1.75$. Поскольку оба значения равны, числа $\frac{7}{4}$ и $1.75$ равны.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Найдите сумму чисел $-3.7$ и $\frac{5}{4}$. Запишите ответ в виде десятичной дроби.

Для решения задачи сначала преобразуем обыкновенную дробь $\frac{5}{4}$ в десятичную дробь. Дробь $\frac{5}{4}$ равна $1.25$. Далее, складываем $-3.7$ и $1.25$: \n\n$$ -3.7 + 1.25 = -3.7 + 1.25 = -2.45 $$
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Фермер собрал с поля 45 тонн картофеля и 30 тонн моркови. В каком отношении масса моркови составляет массу картофеля?

Для того чтобы найти отношение массы моркови к массе картофеля, разделим массу моркови на массу картофеля. Это будет $\frac{30}{45}$. Сократим дробь, разделив числитель и знаменатель на их наибольший общий делитель, который равен 15: $\frac{30 \div 15}{45 \div 15} = \frac{2}{3}$. Следовательно, отношение массы моркови к массе картофеля равно 2:3.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Вычислите значение выражения: $\frac{4}{5} \cdot \left(-\frac{3}{7}\right) \div \frac{2}{3}$.

Для начала произведём умножение: $\frac{4}{5} \cdot \left(-\frac{3}{7}\right) = \frac{4 \cdot (-3)}{5 \cdot 7} = \frac{-12}{35}$. Далее деление заменяется умножением на обратную дробь: $\frac{-12}{35} \div \frac{2}{3} = \frac{-12}{35} \cdot \frac{3}{2}$. После этого умножаем дроби: $\frac{-12 \cdot 3}{35 \cdot 2} = \frac{-36}{70}$. Сократим дробь: $\frac{-36}{70} = \frac{-18}{35}$. Таким образом, значение выражения равно $\frac{-18}{35}$.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Решите уравнение: $\frac{3}{x + 2} + 4 = \frac{5}{x + 2} - 2$

Для решения уравнения приведём дроби к общему знаменателю и сравняем их: \n\n1. Преобразуем уравнение: \n$$\frac{3}{x + 2} + 4 = \frac{5}{x + 2} - 2$$\n\n2. Перенесём все члены с дробями в одну сторону: \n$$\frac{3}{x + 2} - \frac{5}{x + 2} = -4 - 2$$\n\n3. Упростим уравнение: \n$$\frac{3 - 5}{x + 2} = -6$$\n\n4. Это выражается как: \n$$\frac{-2}{x + 2} = -6$$\n\n5. Решим уравнение, домножив обе стороны на $(x + 2)$: \n$$-2 = -6(x + 2)$$\n\n6. Раскроем скобки: \n$$-2 = -6x - 12$$\n\n7. Решим относительно $x$: \n$$6x = -12 + 2$$\n$$6x = -10$$\n$$x = -\frac{10}{6} = -\frac{5}{3}$$
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

In [384]:
for task in tasks:
    question = str(task["original"]).strip().replace("\\\\", "\\")
    solution = str(task["solution"]).strip().replace("\\\\", "\\")

    res = question + "\n\n" + solution + "\n" + 100*"~"
    display(Markdown(res))

Заменив звездочку соответствующей цифрой, заполните таблицу.

|  | Числа, <br> делящиеся <br> на 2 | Числа, <br> делящиеся <br> на 5 | Числа, <br> делящиеся <br> на 10 | Числа, <br> делящиеся <br> на 3 | Числа, <br> делящиеся <br> на 9 |
| :---: | :---: | :---: | :---: | :---: | :---: |
| $49^{*}$ |  |  |  |  |  |
| $83^{*}$ |  |  |  |  |  |

None
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Из чисел $1,2,6,7,13,15,20,41,49$ выпишите числа, имеющие:

1) только один делитель;
2) только два делителя;
3) более двух делителей.

None
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Каковы признаки делимости чисел на 2 ; на 5 ; на 9 ; на 3 ?

None
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Какое число называют делителем данного числа, а какое - его кратным? Приведите примеры.

None
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Сколько делителей имеют простые числа? Приведите примеры.

None
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

...

Сравните дроби:
1) 0,25 и $\frac{3}{4}$;
2) 0,5 и $\frac{1}{4}$;
3) $\frac{5}{6}$ и 0,4 ;
4) 0,6 и $\frac{4}{5}$;
5) 0,7 и $\frac{73}{100}$;
6) 0,2 и $\frac{9}{50}$.

None
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Не выполняя вычислений, из данных выражений составьте верное числовое равенство:
$0,4+\frac{1}{8} ;$
$\frac{3}{4}+0,8 ;$
$\frac{3}{20}+0,25 ;$
$\frac{6}{25}+0,5 ;$
$0,15+\frac{1}{4} ;$
$0,24+\frac{1}{2} ;$
$\frac{2}{5}+0,125 ;$
$0,75+\frac{4}{5}$.

None
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

В фермерском хозяйстве $\frac{3}{5}$ всего поля засеяли пшеницей, а $\frac{7}{20}$ поля заняли овощами. Площадь участка, засеянного пшеницей, на 17 га больше площади участка, занятого овощами. Найдите площадь всего поля фермерского хозяйства.

None
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Вычислите рациональным способом:

1) $\frac{0,2+0,4+0,6+0,8+1+1,2}{1,2+1,4+1,6+1,8+2+2,2+2,4+2,6+2,8+3}$;
2) $\left(\frac{2}{3}+\frac{3}{4}+\frac{4}{5}\right)+\left(\frac{1}{6}+\frac{2}{7}+\frac{3}{8}\right)+\left(\frac{5}{8}+\frac{5}{7}+\frac{5}{6}\right)+\left(\frac{1}{5}+\frac{1}{4}+\frac{1}{3}\right)$.

None
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Решите уравнение:
1) $\frac{1}{x}+\frac{1}{1 \frac{2}{7} x}=\frac{4}{9}$;
2) $\frac{2}{1 \frac{3}{5} x}-\frac{1}{x}=\frac{1}{12}$;
3) $\frac{2}{1 \frac{1}{3} x}-\frac{1}{x}=\frac{1}{4}$.

None
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

In [391]:
# 📦 одна установка, если ещё не ставил
# !pip install markdown termcolor tqdm

import re, logging, markdown, json
from termcolor import colored
from tqdm import tqdm

# ────────────────────────────────────────────────────────────────
# 1. упрощённый валидатор Markdown-строки
def quick_md_check(text: str) -> list[str]:
    """
    Возвращает список найденных проблем.
    Если список пустой — Markdown выглядит корректно.
    """
    problems = []

    # непарные тройные бэктики
    if text.count("```") % 2:
        problems.append("непарное число ``` (блок кода не закрыт)")

    # заголовок без пробела `##Text`
    if re.search(r"^(#{1,6})(\S)", text, flags=re.MULTILINE):
        problems.append("нет пробела после # в заголовке")

    # непарные $…$ (для формул)
    if text.count("$") % 2:
        problems.append("непарное число $ (формула не закрыта)")

    # попытка прогнать через markdown → HTML
    try:
        markdown.markdown(text, extensions=["fenced_code", "tables"])
    except Exception as e:
        problems.append(f"Markdown-parser exception: {e}")

    return problems


In [392]:
# ────────────────────────────────────────────────────────────────
# 2. главная функция-чекер
def check_tasks_format(tasks: list[dict], log_file: str | None = "broken_tasks.log"):
    broken = 0

    # логгер
    logging.basicConfig(
        filename=log_file,
        level=logging.INFO,
        filemode="w",
        format="%(message)s"
    )

    for idx, task in enumerate(tqdm(tasks, desc="Проверка")):
        md_text = task["question"].replace("\\\\", "\\")  # убираем двойные слэши
        errors = quick_md_check(md_text)

        if errors:
            broken += 1
            msg = f"[{idx}] ❌ {errors} | {md_text[:60]}..."
            logging.info(msg)
            print(colored(msg, "red"))
        else:
            print(colored(f"[{idx}] ✅ OK", "green"))

    print(colored(f"\n🔎 Завершено. Поломанных задач: {broken} из {len(tasks)}", "cyan"))
    if broken:
        print(colored(f"Детали записаны в {log_file}", "yellow"))


In [393]:
check_tasks_format(new_tasks)

Проверка: 100%|██████████| 33/33 [00:00<00:00, 543.07it/s]

[0] ✅ OK
[1] ✅ OK
[2] ✅ OK
[3] ✅ OK
[4] ✅ OK
[5] ✅ OK
[6] ✅ OK
[7] ✅ OK
[8] ✅ OK
[9] ✅ OK
[10] ✅ OK
[11] ✅ OK
[12] ✅ OK
[13] ✅ OK
[14] ✅ OK
[15] ✅ OK
[16] ✅ OK
[17] ✅ OK
[18] ✅ OK
[19] ✅ OK
[20] ✅ OK
[21] ✅ OK
[22] ✅ OK
[23] ✅ OK
[24] ✅ OK
[25] ✅ OK
[26] ✅ OK
[27] ✅ OK
[28] ✅ OK
[29] ✅ OK
[30] ✅ OK
[31] ✅ OK
[32] ✅ OK

🔎 Завершено. Поломанных задач: 0 из 33
